In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("atp_matches_2023.csv")
df

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2023-9900,United Cup,Hard,18,A,20230102,300,126203,3.0,NaN,...,62.0,47.0,15.0,12.0,9.0,9.0,9.0,3355.0,16.0,2375.0
1,2023-9900,United Cup,Hard,18,A,20230102,299,126207,NaN,NaN,...,12.0,8.0,3.0,4.0,1.0,3.0,19.0,2000.0,23.0,1865.0
2,2023-9900,United Cup,Hard,18,A,20230102,296,126203,3.0,NaN,...,62.0,51.0,7.0,12.0,2.0,2.0,9.0,3355.0,10.0,2905.0
3,2023-9900,United Cup,Hard,18,A,20230102,295,126207,NaN,NaN,...,41.0,26.0,12.0,9.0,6.0,9.0,19.0,2000.0,245.0,220.0
4,2023-9900,United Cup,Hard,18,A,20230102,292,126774,1.0,NaN,...,58.0,48.0,18.0,16.0,1.0,2.0,4.0,5550.0,16.0,2375.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2981,2023-M-DC-2023-WG2-PO-RSA-LUX-01,Davis Cup WG2 PO: RSA vs LUX,NaN,4,D,20230204,5,202335,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1717.0,1.0
2982,2023-M-DC-2023-WG2-PO-TUN-CYP-01,Davis Cup WG2 PO: TUN vs CYP,NaN,4,D,20230203,1,117365,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,990.0,11.0,279.0,190.0
2983,2023-M-DC-2023-WG2-PO-TUN-CYP-01,Davis Cup WG2 PO: TUN vs CYP,NaN,4,D,20230203,2,121411,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,364.0,131.0,894.0,15.0
2984,2023-M-DC-2023-WG2-PO-TUN-CYP-01,Davis Cup WG2 PO: TUN vs CYP,NaN,4,D,20230203,4,144949,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,894.0,15.0,285.0,184.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2986 entries, 0 to 2985
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   tourney_id          2986 non-null   object 
 1   tourney_name        2986 non-null   object 
 2   surface             2933 non-null   object 
 3   draw_size           2986 non-null   int64  
 4   tourney_level       2986 non-null   object 
 5   tourney_date        2986 non-null   int64  
 6   match_num           2986 non-null   int64  
 7   winner_id           2986 non-null   int64  
 8   winner_seed         1250 non-null   float64
 9   winner_entry        473 non-null    object 
 10  winner_name         2986 non-null   object 
 11  winner_hand         2986 non-null   object 
 12  winner_ht           2969 non-null   float64
 13  winner_ioc          2986 non-null   object 
 14  winner_age          2986 non-null   float64
 15  loser_id            2986 non-null   int64  
 16  loser_

In [5]:

import numpy as np

# Player-level rows from winners
winner_df = df.rename(columns={
    'winner_name': 'player_name',
    'w_ace': 'ace',
    'w_df': 'df',
    'w_svpt': 'serve_pts',
    'w_1stWon': 'first_serve_won',
    'w_2ndWon': 'second_serve_won',
}).loc[:, ['player_name', 'ace', 'df', 'serve_pts', 'first_serve_won', 'second_serve_won']]

# Player-level rows from losers
loser_df = df.rename(columns={
    'loser_name': 'player_name',
    'l_ace': 'ace',
    'l_df': 'df',
    'l_svpt': 'serve_pts',
    'l_1stWon': 'first_serve_won',
    'l_2ndWon': 'second_serve_won',
}).loc[:, ['player_name', 'ace', 'df', 'serve_pts', 'first_serve_won', 'second_serve_won']]

# Combine
player_df = pd.concat([winner_df, loser_df], ignore_index=True)
player_df = player_df.dropna(subset=['serve_pts'])  # remove rows with missing serve data


In [6]:
# Per-match rates
player_df['ace_pct'] = player_df['ace'] / player_df['serve_pts']
player_df['df_pct'] = player_df['df'] / player_df['serve_pts']
player_df['1st_won_pct'] = player_df['first_serve_won'] / player_df['serve_pts']
player_df['2nd_won_pct'] = player_df['second_serve_won'] / player_df['serve_pts']

# Aggregate per player
player_stats = player_df.groupby('player_name').agg({
    'ace_pct': 'mean',
    'df_pct': 'mean',
    '1st_won_pct': 'mean',
    '2nd_won_pct': 'mean'
}).reset_index()


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = player_stats.drop(columns='player_name')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42)
player_stats['style_cluster'] = kmeans.fit_predict(X_scaled)
